# Inverted Disagreement Asymmetry Test - Demo\n\nThis demo notebook loads the mini demo data and shows basic information about the inverted disagreement asymmetry experiment.

In [1]:
# Install dependencies - following aii-colab skill pattern\nimport subprocess, sys\ndef _pip(*args):\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])\n\n# Core packages (pre-installed on Colab, install locally to match Colab env)\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [2]:
# Imports\nimport json\nfrom pathlib import Path\nimport numpy as np\n

In [3]:
# Data loading helper - using the GitHub URL pattern with local fallback\nGITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-2f324c-disagreement-asymmetry-oracle/main/round-2/experiment-1/demo/mini_demo_data.json"\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception: pass\n    local = Path("mini_demo_data.json")\n    if local.exists(): return json.loads(local.read_text())\n    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
# Load and inspect data\ndata = load_data()\nprint(f"Loaded demo data with {len(data['datasets'])} datasets")\nfor dataset in data['datasets']:\n    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")\n\n# Show first example from first dataset\nfirst_example = data['datasets'][0]['examples'][0]\nprint(f"\nFirst example:\n")\nprint(f"  Input: {first_example['input'][:100]}...")\nprint(f"  Output: {first_example['output']}")\nprint(f"  Model A prediction: {first_example['predict_model_a']}")\nprint(f"  Model B prediction: {first_example['predict_model_b']}")\nprint(f"  Inverted rule prediction: {first_example['predict_inverted_rule']}")\nprint(f"  Correct: {first_example['predict_correct']}")

In [5]:
# Simple visualization\nimport matplotlib.pyplot as plt\n%matplotlib inline\n\n# Count predictions across all examples\nall_examples = []\nfor dataset in data['datasets']:\n    all_examples.extend(dataset['examples'])\n\ninverted_predictions = [ex['predict_inverted_rule'] for ex in all_examples]\ncorrect_predictions = [ex['predict_correct'] for ex in all_examples]\n\n# Convert to numpy arrays for easier handling\ninverted_pred_np = np.array([int(p) for p in inverted_predictions])\ncorrect_np = np.array([1 if c == 'True' else 0 for c in correct_predictions])\n\n# Calculate accuracy\naccuracy = np.mean(inverted_pred_np == correct_np)\nprint(f"\nInverted rule accuracy: {accuracy:.3f}")\n\n# Create a simple bar chart\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))\n\n# Prediction distribution\nunique, counts = np.unique(inverted_pred_np, return_counts=True)\nax1.bar(unique, counts)\nax1.set_title('Distribution of Inverted Rule Predictions')\nax1.set_xlabel('Prediction (0 or 1)')\nax1.set_ylabel('Count')\n\n# Accuracy by prediction\nacc_by_pred = []\nfor pred in [0, 1]:\n    mask = (inverted_pred_np == pred)\n    if np.sum(mask) > 0:\n        acc_by_pred.append(np.mean(correct_np[mask]))\n    else:\n        acc_by_pred.append(0)\n\nax2.bar([0, 1], acc_by_pred)\nax2.set_title('Accuracy by Prediction Value')\nax2.set_xlabel('Prediction Value')\nax2.set_ylabel('Accuracy')\nax2.set_ylim(0, 1)\n\nplt.tight_layout()\nplt.show()